In [0]:
dbutils.widgets.removeAll()
dbutils.widgets.text("catalog", "adventure_works")
dbutils.widgets.text("schema_bronze", "bronze")
dbutils.widgets.text("schema_silver", "silver")

catalog = dbutils.widgets.get("catalog")
schema_bronze = dbutils.widgets.get("schema_bronze")
schema_silver = dbutils.widgets.get("schema_silver")


In [0]:
df_customer_bronze = spark.table(f"{catalog}.{schema_bronze}.customer")
df_product_bronze = spark.table(f"{catalog}.{schema_bronze}.product")
df_so_detail_bronze = spark.table(f"{catalog}.{schema_bronze}.so_detail")
df_so_header_bronze = spark.table(f"{catalog}.{schema_bronze}.so_header")
df_territory_bronze = spark.table(f"{catalog}.{schema_bronze}.territory")


In [0]:
df_customer_silver = (
    df_customer_bronze
    .dropDuplicates(["CustomerID"])
    .withColumnRenamed("CustomerID", "customer_id")
    .withColumnRenamed("PersonID", "person_id")
    .withColumnRenamed("StoreID", "store_id")
    .withColumnRenamed("TerritoryID", "territory_id")
)

df_product_silver = (
    df_product_bronze
    .withColumnRenamed("ProductID", "product_id")
    .withColumn("list_price", col("ListPrice").cast("double"))
    .withColumn("standard_cost", col("StandardCost").cast("double"))
)

df_so_header_silver = (
    df_so_header_bronze
    .withColumnRenamed("SalesOrderID", "salesorder_id")
    .withColumn("OrderDate", to_date(col("OrderDate")))
    .withColumn("DueDate", to_date(col("DueDate")))
    .withColumn("ShipDate", to_date(col("ShipDate")))
)

df_so_detail_silver = (
    df_so_detail_bronze
    .withColumnRenamed("SalesOrderID", "salesorder_id")
    .withColumnRenamed("SalesOrderDetailID", "salesorder_detail_id")
)

df_territory_silver = (
    df_territory_bronze
    .withColumnRenamed("TerritoryID", "territory_id")
    .withColumnRenamed("Group", "territory_group")
)


In [0]:
df_customer_silver.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{schema_silver}.customer")
df_product_silver.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{schema_silver}.product")
df_so_header_silver.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{schema_silver}.so_header")
df_so_detail_silver.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{schema_silver}.so_detail")
df_territory_silver.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{schema_silver}.territory")
